In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df = pd.read_csv(path + "/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Distance_km')
plt.xlabel('Preparation_Time_min')
plt.ylabel('Distance_km')
plt.show()

In [ ]:
# Task 1: Write your code here:
if 'Order_ID' in df.columns:
    df = df.drop(columns=['Order_ID'])
    print("Order_ID column dropped.")


In [ ]:
# Task 2: Write your code here:

for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values handled.")
print(df.isnull().sum())

In [ ]:
# Task 3: Write your code here:
initial_len = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_len - len(df)} duplicate rows.")

In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, drop_first=True)
print("Categorical variables encoded.")
display(df.head())

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()

# We scale everything except the target 'delivery_time' usually,

# Here we scale the feature.
target = 'delivery_time'
features = [col for col in df.columns if col != target]

df[features] = scaler.fit_transform(df[features])
print("Features scaled.")

In [ ]:
# Task 6: Write your code here:
skewness = df['Delivery_Time'].skew()
print(f"Target Skewness: {skewness}")

if abs(skewness) > 1:
    print("The target is highly skewed (imbalanced distribution).")
elif abs(skewness) > 0.5:
    print("The target is moderately skewed.")
else:
    print("The target is approximately symmetric (balanced).")

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

# To store feature importances for Part 4
feature_importances = np.zeros(X.shape[1])
# To store predictions for Part 4
y_pred_all = []
y_true_all = []

for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Task 3: Train RandomForest
    model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs= -1, max_depth= 17, min_samples_split=5)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Task 4: Evaluate using MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    # Accumulate data for Part 4 plots
    feature_importances += model.feature_importances_
    y_pred_all.extend(y_pred)
    y_true_all.extend(y_test)

    print(f"Fold {fold+1} MAE: {mae:.4f}")

# Task 5: Average score
average_mae = np.mean(mae_scores)
print(f"\nAverage MAE across 5 folds: {average_mae:.4f}")

In [ ]:
# Task 1: Write your code here:

avg_importance = feature_importances / 5


fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': avg_importance})
fi_df = fi_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df)
plt.title('Top 10 Feature Importance (Random Forest)')
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
sns.histplot(y_pred_all, kde=True, color='orange', label='Predicted')
sns.histplot(y_true_all, kde=True, color='blue', alpha=0.3, label='Actual')
plt.title('Distribution of Predicted vs Actual Delivery Time')
plt.legend()
plt.show()

In [ ]:
# Task Bonus: Write your code here: